In [8]:
# Instalación
# Instalar dependencias
# ! pip install librosa
# ! pip install audioread soundfile   # backends de audio

# En Windows: descargar ffmpeg desde https://ffmpeg.org
# y añadirlo al PATH del sistema
import sys
!{sys.executable} -m pip install librosa audioread soundfile

# Script completo para el corpus
import librosa
import numpy as np
import os

KEY_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
MAJOR_P = [6.35,2.23,3.48,2.33,4.38,4.09,2.52,5.19,2.39,3.66,2.29,2.88]
MINOR_P = [6.33,2.68,3.52,5.38,2.60,3.53,2.54,4.75,3.98,2.69,3.34,3.17]

def detect_key(y, sr):
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr).mean(axis=1)
    best_score, best_key = -1, ''
    for i in range(12):
        rot = np.roll(chroma, -i)
        for profile, mode in [(MAJOR_P,'Major'),(MINOR_P,'Minor')]:
            score = np.corrcoef(rot, profile)[0,1]
            if score > best_score:
                best_score, best_key = score, f'{KEY_NAMES[i]} {mode}'
    return best_key

def analyze(path):
    y, sr = librosa.load(path, duration=90, mono=True)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    key      = detect_key(y, sr)
    energy   = min(100, int(np.sqrt(np.mean(y**2)) * 1000))
    return round(float(tempo)), key, energy

folder = '../data/raw/Mp3-Top15-songs'
for f in sorted(os.listdir(folder)):
    if f.endswith('.mp3'):
        bpm, key, energy = analyze(os.path.join(folder, f))
        print(f'{f:<50} BPM:{bpm:>4}  Key:{key:<12}  Energy:{energy:>3}/100')


C:\Users\CRIS\AppData\Local\Temp\ipykernel_24340\1395086243.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return round(float(tempo)), key, energy


505 - Arctic Monkeys.mp3                           BPM: 144  Key:E Minor       Energy:100/100
BTS (방탄소년단) 'Butter' Official MV.mp3               BPM: 112  Key:G# Major      Energy:100/100
BTS (방탄소년단) 'Dynamite' Official MV.mp3             BPM: 112  Key:E Major       Energy:100/100
BTS (방탄소년단) 'FAKE LOVE' Official MV.mp3            BPM: 152  Key:D Minor       Energy:100/100
CD1 - 22 - Creep (Live).mp3                        BPM:  99  Key:G Major       Energy: 95/100
Chappell Roan - Good Luck, Babe! (Official Lyric Video) (1).mp3 BPM: 117  Key:D Major       Energy:100/100
Chappell Roan - Good Luck, Babe! (Official Lyric Video).mp3 BPM: 117  Key:D Major       Energy:100/100
Coldplay X BTS - My Universe (Official Video).mp3  BPM: 103  Key:E Major       Energy:100/100
Mitski - My Love Mine All Mine (Official Lyric Video).mp3 BPM: 112  Key:A Major       Energy:100/100
No Surprises.mp3                                   BPM: 152  Key:F Major       Energy:100/100
Sabrina Carpenter - Espresso (O